# TikTok Style Lyric Video Creator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/editor/creative/tiktok_style_lyric_video_creator.ipynb)

### Create TikTok/Reels/Shorts-Ready Content

Turn any music video into viral-ready vertical clips optimized for social media.

This notebook automates the complete workflow:
- AI identifies the catchiest segments (chorus with build-up)
- Generates vertical 9:16 backgrounds for mobile viewing
- Creates mobile-optimized captions with high contrast
- Adds pre-roll, CTA, and post-roll for professional polish
- Outputs 15-60 second clips ready to upload

All powered by **VideoDB's Editor SDK** - built for social media from the ground up.

---

## 🚀 Setup

Install the VideoDB SDK to unlock AI-powered video editing and generation capabilities.

In [4]:
!pip install -q videodb

### Connect to VideoDB

Authenticate with your API key to access the full suite of video intelligence and editing tools.

In [5]:
from getpass import getpass
import videodb
import os

# Connect to VideoDB
api_key = getpass("🔑 Enter your VideoDB API Key: ")
os.environ["VIDEO_DB_API_KEY"] = api_key

conn = videodb.connect()
coll = conn.get_collection()

print("✅ Connected to VideoDB!")

🔑 Enter your VideoDB API Key: ··········
✅ Connected to VideoDB!


---

## 🎵 Step 1: Upload Your Music Video

Enter the URL of any music video you want to transform into short-form content. The system will upload and prepare it for processing.

In [6]:
video_url = "https://youtu.be/V9PVRfjEBTI"

video = coll.upload(url=video_url)
print(video)

Video(id=m-z-019f8f79-13bf-7203-91e4-1e3b840e4674, collection_id=c-81fc6459-fe30-44ac-8c5b-ea0898c2e152, stream_url=https://play.videodb.io/v1/e0e0075f-ca39-44e7-97fd-2de2315be41f.m3u8, player_url=https://player.videodb.io/watch?v=0hfLKJRNwXw, name=Billie Eilish - BIRDS OF A FEATHER (Official Music Video), description=None, thumbnail_url=None, length=230.0)


In [7]:
video.play()

---

## 🎨 Step 2: Define Your Visual Aesthetic

Describe the visual vibe you want for your short-form content. This guides the AI when generating vertical background images.

**Mobile-First Examples:**
- "Calm and serene vibe. Ocean, beach, sunsets and peace"
- "Urban nightlife, neon signs, city energy"
- "Minimalist, pastel colors, dreamy gradients"
- "Bold and vibrant, pop art style, eye-catching"

Remember: Short-form content needs to grab attention fast, so be bold with your aesthetic choices!

In [8]:
user_request = "Romantic & peaceful"
user_request

'Romantic & peaceful'

---

## 📝 Step 3: Create a Timed Transcript Artifact

Create a timed transcript artifact with server-provided word text and timestamp ranges. These timed words support lyric planning while keeping timing aligned with the source video.

In [11]:
print("Creating timed transcript artifact... this might take a moment.")

transcript_understanding = video.understand(
    analyzers=[
        {
            "type": "spoken_words",
            "name": "transcript",
        },
    ],
)
transcript_understanding.wait_until_complete()

transcript_analyzer = transcript_understanding.get_analyzer("transcript")



Creating timed transcript artifact... this might take a moment.


### Prepare Timed Transcript Words

Use the transcript artifact to prepare:
- **Timed words** — Server-provided word text and timestamp ranges for lyric alignment
- **Plain transcript text** — For AI analysis of song structure

In [12]:
from math import isfinite

transcript_output = transcript_analyzer.get_output()
raw_entries = []
transcript_parts = []
source_order = 0

for scene in transcript_output.get("scenes", []):
    data = scene.get("data", {})

    scene_text = data.get("text")
    if isinstance(scene_text, str) and scene_text.strip():
        transcript_parts.append(scene_text.strip())

    for word in data.get("words", []):
        text = word.get("text")
        normalized_text = text.strip() if isinstance(text, str) else ""
        silence_marker = normalized_text.strip("[]()<>").strip().casefold()
        start = word.get("start")
        end = word.get("end")

        if (
            normalized_text
            and silence_marker not in {"-", "silence"}
            and isinstance(start, (int, float))
            and not isinstance(start, bool)
            and isinstance(end, (int, float))
            and not isinstance(end, bool)
            and isfinite(start)
            and isfinite(end)
            and 0 <= start < end
        ):
            raw_entry = {
                "start": start,
                "end": end,
                "text": normalized_text,
                "source_order": source_order,
            }
            raw_entries.append(raw_entry)

        source_order += 1

raw_entries.sort(
    key=lambda entry: (entry["start"], entry["end"], entry["source_order"])
)

transcript_timed = []
for index, entry in enumerate(raw_entries, start=1):
    timed_entry = {
        "id": f"entry_{index:04d}",
        "start": entry["start"],
        "end": entry["end"],
        "text": entry["text"],
    }
    transcript_timed.append(timed_entry)

transcript_entries_by_id = {
    entry["id"]: entry
    for entry in transcript_timed
}
transcript_entry_positions = {
    entry["id"]: index
    for index, entry in enumerate(transcript_timed)
}

transcript_text = "\n".join(transcript_parts)
if not transcript_text:
    transcript_text = " ".join(entry["text"] for entry in transcript_timed)

print(f"Prepared {len(transcript_timed)} timed transcript words.")
print(transcript_timed[:3])

Prepared 245 timed transcript words.
[{'id': 'entry_0001', 'start': 3.84, 'end': 4.04, 'text': 'I'}, {'id': 'entry_0002', 'start': 4.04, 'end': 4.36, 'text': 'want'}, {'id': 'entry_0003', 'start': 4.36, 'end': 4.64, 'text': 'you'}]


In [13]:
print(transcript_text)

I want you to stay
stay till I'm in the grave
grave Till I run away
away dead and bury Till I can't
can't skin you carry if you go I'm going through
through
Cuz it was always you
you
and if I'm turning
turning blue
please don't
don't save me Nothing left to
to lose without
my baby
Birds of a feather we
should stick together I know
know I said I'd never
think
I wasn't better alone
alone can't change the weather Might not be forever but if it's
it's forever it's even better and
and I don't know
know but I'll
I'll cry I
I don't
think I could
could love you more
might not be long but baby
baby I
I love you
you till the day that I die
Till the
the day that I die
die
Till the
the light is my eyes
eyes
Till
Till the day that I
I die
you to see
how you look to me
you wouldn't believe
believe if I told you you
you would keep the
the compliments I throw you but you're so full of
of
tell me
so you don't see
see it your mind's
say you want to quit don't be
stupid
I don't know
cry
I don't think I c

---

## 🎯 Step 4: AI-Powered Segment Selection & Visual Generation

This is where the AI works its magic! The LLM analyzes the entire song to:

### Find Viral-Worthy Segments
- **Identify the chorus** — The most catchy, repeatable part
- **Include build-up** — 2-4 lines before the chorus for context
- **Add wind-down** — 2-4 lines after for smooth resolution
- **Ensure completeness** — Each segment feels like a mini-story with beginning, climax (chorus), and end
- **Prioritize quality** — Better to have 1 amazing clip than 3 mediocre ones

### Create Mobile-Optimized Visuals
For each selected segment, the AI generates:
- **Vertical image prompts (9:16)** — Designed for portrait mobile viewing
- **Central negative space** — Clear area for text overlay
- **Font color selection** — Maximum contrast for readability on small screens
- **Lyric segmentation** — Optimized line length when timed words permit it
- **Artifact-word-aligned lines** — Every displayed line is derived from selected timed words

### What Makes a Segment "Catchy"?
The AI looks for:
- The chorus/hook with emotional peak
- Quotable, shareable lyrics
- Energy shifts and beat drops
- Viral potential (trends, dances, duets)
- Memorable vocal moments

The output is 1-3 short segments, each 15-60 seconds long, ready to go viral!

In [15]:
import json

prompt = f"""
You are a viral social media content strategist specializing in short-form vertical video. Your task is to identify the most engaging, catchy, and shareable segments from a music video to create TikTok/Reels/Shorts content.

# INPUT DATA

1. Full Transcript Text (may contain ASR errors/missing punctuation):
{transcript_text}

2. Timed Transcript Words (JSON array with local id/start/end/text fields):
{json.dumps(transcript_timed, ensure_ascii=False)}

3. Video Metadata:
   - Name: {video.name}
   - Duration: {video.length} seconds
   - User Style Request: {user_request}

# SEGMENT SELECTION CRITERIA

Identify **1-3 high-quality segments** (prioritize quality over quantity). Each segment should:

## What Makes a Segment "Catchy"?

**PRIMARY FOCUS: Chorus/Hook with Proper Framing**

Segments MUST be built around the chorus with these components:
1. **Build-up (Pre-Chorus/Verse End)**: 2-4 lines leading into the chorus that create anticipation
2. **The Chorus/Hook**: The main catchy, repetitive, memorable section
3. **Wind-down (Post-Chorus)**: 2-4 lines after the chorus that provide resolution

The segment structure should be: **Ramp Up → Peak (Chorus) → Ramp Down**

Additional qualities that enhance a chorus segment:
- **Emotional Peak**: Most intense, emotionally charged moment (climax, drop, powerful vocals)
- **Quotable Lines**: Lyrics that are relatable, funny, profound, or highly shareable
- **Viral Potential**: Lyrics that could inspire trends, dances, duets, or memes
- **Energy Shift**: Dramatic beat drop, tempo change, or dynamic transition in/around the chorus
- **Memorable Moment**: Distinctive vocal run, ad-lib, or production element that makes the chorus special

**Non-Negotiable Rules:**
- The chorus is the centerpiece - always include it
- Never start directly on the first chorus line - include the approach
- Never end directly after the last chorus line - include the exit
- Build-up and wind-down are MANDATORY for professional feel

## Segment Requirements
- **Duration**: 15-60 seconds per segment (optimal: 20-45 seconds for retention)
- **Focus on Chorus**: Segments should CENTER around the chorus/hook - this is the primary target
- **Build-up REQUIRED**: MUST include the build-up/lead-in before the chorus (last 2-4 lines of pre-chorus or verse)
- **Wind-down REQUIRED**: MUST include the wind-down/resolution after the chorus (first 2-4 lines after chorus ends)
- **No Abrupt Starts**: Never start directly on the main chorus line - include runway space before it
- **No Abrupt Ends**: Never cut immediately when the chorus ends - include graceful exit
- **Ramp Up & Ramp Down**: The segment should feel like a complete emotional arc with natural entry and exit
- **Completeness**: Each segment should feel like a complete mini-story with beginning, climax (chorus), and resolution
- **Standalone Quality**: Segment must work independently and feel professionally edited, not chopped

## Quality Over Quantity
- If only 1 truly excellent segment exists, return only that one
- Do NOT force multiple segments if the song doesn't naturally have them
- Better to have 1 amazing 30-second clip than 3 mediocre ones

# LYRIC PROCESSING

## Timed Word Handling
- Treat every timed transcript word as indivisible
- Do NOT rewrite ASR text, split words, interpolate timestamps, or create synthetic records
- A lyric line may select only a consecutive range of local entry IDs in chronological order
- Keep the 3-8 word and 20-character goals only when whole-word grouping permits them
- Preserve every word intact; timing fidelity takes priority

## Timing and Entry Selection
- Each line returns only the first and last selected entry IDs
- Lines and stanzas in a segment must be ordered, adjacent, non-overlapping, and must not reuse IDs
- Include every intervening selectable entry between a segment's first and last IDs; do not skip entries
- The notebook derives all lyric text, line timing, stanza bounds, and segment bounds locally from the selected words

# IMAGE GENERATION REQUIREMENTS

For EACH stanza within EACH segment, create an "image_prompt" following these rules:

## Short-Form Specific Considerations
- **Vertical aspect ratio**: 9:16 (portrait orientation for mobile)
- **Mobile-first design**: Images should look compelling on small phone screens
- **Attention-grabbing**: Short-form content needs more visual impact than full videos
- **Text readability**: Even more critical in vertical format with limited screen space

## Independence & Self-Containment
- Each prompt must be FULLY SELF-CONTAINED with complete scene descriptions
- NEVER use references like "previous image", "same as before", "similar to", or "continue from"
- Each prompt must work standalone if generated in isolation

## Content & Style
- Honor the user's style request: {user_request}
- Reflect the mood, theme, and emotional tone of the current stanza
- Create visual progression within each segment
- **CRITICAL**: Prompts must NEVER include text, lyrics, words, letters, names, or any written language in the generated image

## Composition for Vertical Video Text Overlay
- **Vertical framing**: Design for 9:16 portrait orientation
- Keep **center vertical strip** relatively clear for text overlay
- Text typically appears in middle-to-upper-middle area on mobile
- Avoid busy details in the central vertical zone
- Can have more detail at top/bottom edges where text is less likely
- Use balanced, mobile-optimized compositions
- Images should be visually striking but NOT overly distracting

## Visual Consistency Within Each Segment
Each segment should have internal visual consistency:
- **Color palette**: Harmonious colors within the segment
- **Artistic style**: Consistent style throughout the segment
- **Mood/atmosphere**: Unified emotional tone
- **Composition approach**: Similar framing strategy

Note: Different segments CAN have different visual themes if they represent different song moods.

## Prompt Structure
Each image_prompt should specify:
1. Scene/subject matter relevant to the stanza
2. Emotional mood and atmosphere matching the segment's energy
3. Lighting conditions and color tone
4. Artistic style (aligned with user_request)
5. **Vertical composition notes** (e.g., "portrait orientation, centered vertical negative space, detailed top and bottom thirds")
6. Mobile-optimized visual impact

# FONT COLOR SELECTION

For EACH stanza, select a "font_color" ensuring optimal mobile readability:

## Available Colors
Choose from this list ONLY:
- White: #FFFFFF (for dark backgrounds)
- Black: #000000 (for light backgrounds)
- Yellow: #FFFF00 (for dark backgrounds, extremely high visibility on mobile)
- Dark Blue: #00008B (for light backgrounds)
- Dark Green: #006400 (for light backgrounds)

## Selection Guidelines
- **Light/bright backgrounds** → Black (#000000), Dark Blue (#00008B), or Dark Green (#006400)
- **Dark/dim backgrounds** → White (#FFFFFF) or Yellow (#FFFF00)
- **Mobile priority**: Ensure MAXIMUM CONTRAST for small screen readability
- Consider that users often view in bright outdoor lighting or dim rooms
- Yellow (#FFFF00) works exceptionally well for high-energy viral content on dark backgrounds

# SEGMENT METADATA

For each segment, provide:

## segment_title
- A catchy, descriptive title for the segment (3-6 words)
- Should hint at what makes this segment special
- Examples: "Explosive Chorus Drop", "Emotional Bridge Moment", "Viral Hook Section"

## Segment bounds
- Segment bounds are derived locally from the first and last selected transcript words
- Do not return numeric bounds or duration

# OUTPUT FORMAT

Return ONLY a valid JSON array with this exact structure:

[
  {{
    "segment_title": "string (catchy 3-6 word title)",
    "why_catchy": "string (1-2 sentence explanation of what makes this segment viral-worthy)",
    "stanzas": [
      {{
        "image_prompt": "string (detailed, self-contained, vertical 9:16 image generation prompt)",
        "font_color": "string (hex code from approved list)",
        "lines": [
          {{
            "first_id": "entry_0001",
            "last_id": "entry_0003"
          }}
        ]
      }}
    ]
  }}
]

Do not return lyric text, numeric start or end values, stanza bounds, segment bounds, or duration.

# CRITICAL REMINDERS

1. **CHORUS-CENTERED STRUCTURE**: Every segment must be built around a chorus with proper build-up and wind-down
2. **NO ABRUPT STARTS**: Always include 2-4 lines BEFORE the chorus begins (pre-chorus/verse ending)
3. **NO ABRUPT ENDS**: Always include 2-4 lines AFTER the chorus ends (post-chorus beginning)
4. **PROFESSIONAL RAMP UP/DOWN**: The segment should feel like a complete emotional journey, not a choppy cut
5. **Quality over quantity**: 1 amazing segment > 3 mediocre ones
6. **Each image_prompt is independent**: No cross-references between prompts
7. **Vertical format**: All image prompts must specify 9:16 portrait orientation
8. **No text in images**: Never include lyrics, words, or letters in image generation prompts
9. **Complete coverage**: Use every intervening word between each segment endpoint
10. **Mobile-first**: Optimize everything for small screen viewing
11. **Honor style request**: All creative decisions must align with: {user_request}
12. **Word integrity**: Keep words intact when compact-line goals conflict with timing fidelity

Output the JSON array only, no additional text.
"""

stanzas = coll.generate_text(
    prompt=prompt,
    model_name="pro",
    response_type="json",
)
print(stanzas)

{'output': {'segment_title': 'Till The Day Chorus', 'stanzas': [{'font_color': '#FFFFFF', 'image_prompt': 'Portrait 9:16 vertical scene of a quiet seaside at dusk: a romantic, peaceful atmosphere with a lone couple walking along wet sand, soft pastel sky of rose and lavender, distant gentle waves reflecting warm twilight. Cinematic shallow depth-of-field, soft film grain, and subtle lens flare on the horizon. Composition intentionally leaves a clear vertical center strip for text overlay (minimal details in center), richer textured details in the top third (sky) and bottom third (sand and reflections). Art direction: dreamy, painterly realism with soft edges and harmonious warm-pastel color palette, mobile-first high contrast, romantic & peaceful mood.', 'lines': [{'first_id': 'entry_0117', 'last_id': 'entry_0120'}, {'first_id': 'entry_0121', 'last_id': 'entry_0124'}]}, {'font_color': '#FFFFFF', 'image_prompt': "Portrait 9:16 vertical close-up scene: intimate, peaceful embrace at golde

### Setup: Validation and Rendering Configuration

Before rendering, we set up:
- **Text positioning helper** — Calculates vertical offsets for centered lyrics
- **Entry-ID validation and local rehydration** — Builds lyric lines from selected transcript words
- **Non-mutating stanza preparation** — Keeps artifact-derived timing data intact
- **Timeline configuration** — 608x1080 vertical resolution (9:16 ratio)
- **Timing constants** — Pre-roll, post-roll, and CTA durations

In [17]:
import math
from videodb.editor import (
    VideoAsset, ImageAsset, TextAsset, Font,
    Clip, Track, Timeline, Transition,
    Position, Fit, Offset, Alignment, HorizontalAlignment, VerticalAlignment, Border
)

def calculate_y_offsets(num_lines, line_height=45, timeline_height=1080):
    offsets = []
    start_pixel_offset = -((num_lines - 1) * line_height) / 2
    half_height = timeline_height / 2
    for i in range(num_lines):
        pixel_y = start_pixel_offset + (i * line_height)
        offsets.append(pixel_y / half_height)
    return offsets

def optimize_segment_stanzas(stanzas):
    if not stanzas:
        return []

    return [
        {
            **stanza,
            "lines": [dict(line) for line in stanza["lines"]],
        }
        for stanza in stanzas
    ]


ALLOWED_FONT_COLORS = {
    "#FFFFFF",
    "#000000",
    "#FFFF00",
    "#00008B",
    "#006400",
}


def require_nonblank_string(value, field_name):
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{field_name} must be a nonblank string.")
    return value


def require_exact_keys(value, expected_keys, label):
    if not isinstance(value, dict) or set(value) != expected_keys:
        raise ValueError(f"{label} has an invalid shape.")


def rehydrate_line(line_plan, expected_position):
    require_exact_keys(line_plan, {"first_id", "last_id"}, "Lyric line")
    first_id = require_nonblank_string(line_plan["first_id"], "first_id")
    last_id = require_nonblank_string(line_plan["last_id"], "last_id")
    if first_id not in transcript_entries_by_id:
        raise ValueError(f"Unknown first entry ID: {first_id}")
    if last_id not in transcript_entries_by_id:
        raise ValueError(f"Unknown last entry ID: {last_id}")

    first_position = transcript_entry_positions[first_id]
    last_position = transcript_entry_positions[last_id]
    if first_position > last_position:
        raise ValueError("Lyric line entry IDs are out of order.")
    if expected_position is not None and first_position != expected_position:
        raise ValueError(
            "Lyric lines in a segment must be adjacent, ordered, and non-overlapping."
        )

    entries = transcript_timed[first_position : last_position + 1]
    return (
        {
            "text": " ".join(entry["text"] for entry in entries),
            "start": entries[0]["start"],
            "end": entries[-1]["end"],
        },
        last_position + 1,
    )


raw_segments = stanzas.get("output")
# If the model returns a single segment as a dict, wrap it in a list
if isinstance(raw_segments, dict):
    raw_segments = [raw_segments]

if not isinstance(raw_segments, list) or not raw_segments:
    raise RuntimeError("VideoDB did not return any segment plans.")

segments = []
for segment_number, segment_plan in enumerate(raw_segments, start=1):
    require_exact_keys(
        segment_plan,
        {"segment_title", "why_catchy", "stanzas"},
        f"Segment {segment_number}",
    )
    raw_stanzas = segment_plan["stanzas"]
    if not isinstance(raw_stanzas, list) or not raw_stanzas:
        raise ValueError(f"Segment {segment_number} must contain stanzas.")

    hydrated_stanzas = []
    expected_position = None
    for stanza_number, stanza_plan in enumerate(raw_stanzas, start=1):
        require_exact_keys(
            stanza_plan,
            {"image_prompt", "font_color", "lines"},
            f"Segment {segment_number}, stanza {stanza_number}",
        )
        image_prompt = require_nonblank_string(
            stanza_plan["image_prompt"], "image_prompt"
        )
        font_color = require_nonblank_string(stanza_plan["font_color"], "font_color")
        if font_color not in ALLOWED_FONT_COLORS:
            raise ValueError(f"Unsupported font color: {font_color}")
        line_plans = stanza_plan["lines"]
        if not isinstance(line_plans, list) or not line_plans:
            raise ValueError(
                f"Segment {segment_number}, stanza {stanza_number} must contain lines."
            )

        hydrated_lines = []
        for line_plan in line_plans:
            line, expected_position = rehydrate_line(line_plan, expected_position)
            hydrated_lines.append(line)

        stanza_start = min(line["start"] for line in hydrated_lines)
        stanza_end = max(line["end"] for line in hydrated_lines)
        hydrated_stanzas.append(
            {
                "stanza_start": stanza_start,
                "stanza_end": stanza_end,
                "image_prompt": image_prompt,
                "font_color": font_color,
                "lines": hydrated_lines,
            }
        )

    segment_start = min(stanza["stanza_start"] for stanza in hydrated_stanzas)
    segment_end = max(stanza["stanza_end"] for stanza in hydrated_stanzas)
    if segment_end <= segment_start:
        raise ValueError(f"Segment {segment_number} has an invalid duration.")
    segments.append(
        {
            "segment_title": require_nonblank_string(
                segment_plan["segment_title"], "segment_title"
            ),
            "start_time": segment_start,
            "end_time": segment_end,
            "duration": segment_end - segment_start,
            "why_catchy": require_nonblank_string(
                segment_plan["why_catchy"], "why_catchy"
            ),
            "stanzas": hydrated_stanzas,
        }
    )

TIMELINE_WIDTH = 608
TIMELINE_HEIGHT = 1080
CTA_DURATION = 2.0

### Parallel Image Generation

Generate all background images concurrently for maximum speed. This cell uses ThreadPoolExecutor to create multiple images at once, significantly reducing wait time.

In [18]:
from concurrent.futures import ThreadPoolExecutor, as_completed

image_tasks = []
for seg_idx, segment in enumerate(segments):
    segment["optimized_stanzas"] = optimize_segment_stanzas(segment["stanzas"])

    for stan_idx, s in enumerate(segment["optimized_stanzas"]):
        image_tasks.append({
            "seg_idx": seg_idx,
            "stan_idx": stan_idx,
            "prompt": s["image_prompt"]
        })

def generate_worker(task):
    print(f"Making 9:16 image for Seg {task['seg_idx']+1}, Stanza {task['stan_idx']+1}...")
    img = coll.generate_image(prompt=task["prompt"], aspect_ratio="9:16")
    return task["seg_idx"], task["stan_idx"], img.id

MAX_WORKERS = 10
if image_tasks:
    print(f"Starting parallel make of {len(image_tasks)} images...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(generate_worker, task) for task in image_tasks]

        for future in as_completed(futures):
            try:
                seg_i, stan_i, img_id = future.result()
                segments[seg_i]["optimized_stanzas"][stan_i]["image_id"] = img_id
            except Exception as e:
                print(f"Error making image: {e}")

    print("Full image make complete.")
else:
    print("No segments found to process.")

Starting parallel make of 3 images...
Making 9:16 image for Seg 1, Stanza 1...
Making 9:16 image for Seg 1, Stanza 2...
Making 9:16 image for Seg 1, Stanza 3...
Full image make complete.


---

## 🎬 Step 6: Render Short-Form Videos

The grand finale! For each viral-worthy segment, we build a complete short-form video with:

### Video Structure
1. **5-Second Pre-Roll** — Smooth fade-in build-up before the catchy part begins
2. **Catchy Segment** — The chorus/hook with precisely timed lyric lines (15-60 seconds)
3. **5-Second CTA** — "Visit the channel for the full video" to drive traffic
4. **5-Second Post-Roll** — Fade-out outro for smooth ending

### Multi-Layer Composition

**Layer 1: Audio Track**
- Original music from the source video
- 5-second fade-in/out for professional polish
- Hidden video (opacity=0) so only audio plays

**Layer 2: Background Images**
- AI-generated vertical images for each stanza
- First image extends to beginning (covers pre-roll)
- Smooth fade transitions between images
- Optimized for mobile viewing

**Layer 3: Animated Lyrics**
- Large, bold text (size 48) for mobile readability
- Black borders for maximum contrast
- Timed lyric lines derived from transcript words
- Fade-in animations for smooth appearance
- Centered positioning with calculated vertical offsets

**Layer 4: CTA Text**
- Two-line call-to-action at the end
- Drives viewers to the full video/channel
- Professional fade-in effect

Each segment is rendered as a standalone video, ready to upload to TikTok, Instagram Reels, or YouTube Shorts!

In [19]:
for idx, segment in enumerate(segments):
    print(f"Rendering Segment {idx+1}: {segment['segment_title']}")

    # 0. Timing Calculations
    PRE_ROLL = 5.0
    POST_ROLL = 5.0
    CTA_DURATION = 5.0

    actual_seg_start = segment["start_time"]
    actual_seg_end = segment["end_time"]

    # Calculate where in the OG video we actually start (handling start < 5s)
    timeline_start_og = max(0, actual_seg_start - PRE_ROLL)
    intro_duration = actual_seg_start - timeline_start_og

    # Total audio duration: Intro + Catchy Part + CTA + Outro
    total_audio_duration = (actual_seg_end - timeline_start_og) + CTA_DURATION + POST_ROLL

    stanza_items = segment["optimized_stanzas"]

    timeline = Timeline(conn)
    timeline.resolution = f"{TIMELINE_WIDTH}x{TIMELINE_HEIGHT}"
    timeline.background = "#000000"

    # 1. Audio Track (With 5s Fade In/Out Volume Ramp)
    audio_track = Track(z_index=0)
    audio_track.add_clip(
        start=0.0,
        clip=Clip(
            asset=VideoAsset(id=video.id, start=timeline_start_og, volume=1.0),
            duration=total_audio_duration,
            fit=Fit.crop,
            position=Position.center,
            opacity=0.0,
            transition=Transition(in_="fade", out="fade", duration=5.0)
        ),
    )
    timeline.add_track(audio_track)

    # 2. Background Images Track
    images_track = Track(z_index=1)
    for i, s in enumerate(stanza_items):
        # Apply safety net bypass: Skip image processing if no image was generated (Quota issue)
        if "image_id" not in s or not s["image_id"]:
            continue

        # Calculate local timeline timing
        local_start = s["stanza_start"] - timeline_start_og
        local_end = s["stanza_end"] - timeline_start_og

        # If it's the first image, stretch it back to the start of the timeline (0.0)
        if i == 0:
            local_start = 0.0
            trans_in = Transition(in_="fade", duration=intro_duration)
        else:
            trans_in = Transition(in_="fade", out="fade", duration=0.35)

        duration = max(0.1, local_end - local_start)

        images_track.add_clip(
            start=local_start,
            clip=Clip(
                asset=ImageAsset(id=s["image_id"]),
                duration=duration,
                fit=Fit.crop,
                position=Position.center,
                transition=trans_in
            ),
        )
    timeline.add_track(images_track)

    # 3. Lyrics Track
    lyrics_track = Track(z_index=2)
    LINE_HEIGHT = 45
    lyrics_border = Border(color="#000000", width=1.5)

    for s in stanza_items:
        lines = s.get("lines", [])
        y_offsets = calculate_y_offsets(len(lines), LINE_HEIGHT, TIMELINE_HEIGHT)
        # All lyric timings shifted by timeline_start_og

        for l_idx, line in enumerate(lines):
            local_line_start = line["start"] - timeline_start_og
            line_duration = line["end"] - line["start"]
            line_fade_duration = min(0.5, line_duration / 2)

            text_asset = TextAsset(
                text=line["text"],
                font=Font(family="Bebas Neue", size=48, color="#FFFFFF"),  # Removed weight=700
                border=lyrics_border,
                alignment=Alignment(horizontal=HorizontalAlignment.center, vertical=VerticalAlignment.center)
            )

            lyrics_track.add_clip(
                start=local_line_start,
                clip=Clip(
                    asset=text_asset,
                    duration=line_duration,
                    position=Position.center,
                    offset=Offset(x=0, y=y_offsets[l_idx]),
                    transition=Transition(in_="fade", duration=line_fade_duration)
                )
            )

    # 4. Decoupled CTA Segment (Visit Channel + Full Video as separate centered lines)
    cta_start_time = (actual_seg_end - timeline_start_og)
    cta_lines = ["Visit the channel", "for the full video"]
    cta_y_offsets = calculate_y_offsets(len(cta_lines), line_height=60, timeline_height=TIMELINE_HEIGHT)

    for c_idx, cta_text in enumerate(cta_lines):
        cta_asset = TextAsset(
            text=cta_text,
            font=Font(family="Bebas Neue", size=42, color="#FFFFFF"), # Removed weight=700
            border=lyrics_border,
            alignment=Alignment(horizontal=HorizontalAlignment.center, vertical=VerticalAlignment.center)
        )

        lyrics_track.add_clip(
            start=cta_start_time,
            clip=Clip(
                asset=cta_asset,
                duration=CTA_DURATION,
                position=Position.center,
                offset=Offset(x=0, y=cta_y_offsets[c_idx]),
                transition=Transition(in_="fade", duration=0.5)
            )
        )

    timeline.add_track(lyrics_track)

    # 5. Compile and Generate
    stream_url = timeline.generate_stream()

Rendering Segment 1: Till The Day Chorus


### Preview Your Viral-Ready Content

Watch the final short-form video! It's ready to share on social media and start getting views.

In [20]:
from videodb import play_stream
play_stream(stream_url)

---

## That's a Wrap!

We just transformed a full music video into viral-ready short-form content using AI!

### What we built:
- **AI segment detection** - Found the catchiest moments (chorus-focused)
- **Vertical visuals** - Custom 9:16 backgrounds for mobile
- **Mobile-optimized text** - Large, high-contrast captions
- **Complete shorts** - Pre-roll, catchy segment, CTA, post-roll
- **Social-ready** - Perfect for TikTok, Reels, and Shorts

### The Magic of VideoDB Editor SDK:
- No manual clipping - AI identifies viral moments automatically
- No vertical reformatting - Built for 9:16 from the start
- **Timed lyric sync** - Artifact words are locally rehydrated into lyric lines
- No design work - AI generates on-brand visuals
- Multiple clips from one song - Maximize content output

**Ready to create more?** Upload different songs and experiment with aesthetics!

---

*Made with VideoDB Editor SDK*